In [ ]:
import pandas as pd

df = pd.read_csv('teaching_prices_clean.csv')
print(df.shape)
print(df['data_label'].unique())
df.head()

In [ ]:
rows = []
for item, g in df.groupby('item'):
    first_date, last_date = g['date'].min(), g['date'].max()
    first = g.loc[g['date'] == first_date, 'price'].mean()
    last = g.loc[g['date'] == last_date, 'price'].mean()
    rows.append({
        'item': item, 'n': len(g),
        'mean': g['price'].mean(), 'median': g['price'].median(),
        'min': g['price'].min(), 'max': g['price'].max(),
        'first': first, 'last': last,
        'pct_change': (last / first - 1) * 100,
    })
stats = pd.DataFrame(rows).sort_values('item').round(2)
print(stats.to_string(index=False))

In [ ]:
stats['mean_minus_median'] = (stats['mean'] - stats['median']).round(2)
stats['gap_pct_of_median'] = (stats['mean_minus_median'] / stats['median'] * 100).round(2)
print(stats[['item', 'mean', 'median', 'mean_minus_median', 'gap_pct_of_median']].to_string(index=False))

In [ ]:
staple = df[df['item'] == 'Staple A'].copy()
clean_mean, clean_median = staple['price'].mean(), staple['price'].median()

flagged = staple.copy()
mask = (flagged['date'] == '2026-01-08') & (flagged['city'] == 'City 1')
print('original value:', flagged.loc[mask, 'price'].iloc[0])
flagged.loc[mask, 'price'] = 995.0

print(f"clean  : mean={clean_mean:.2f} median={clean_median:.2f}")
print(f"flagged: mean={flagged['price'].mean():.2f} median={flagged['price'].median():.2f}")
print(f"mean moved {flagged['price'].mean() - clean_mean:+.2f} ({(flagged['price'].mean()/clean_mean-1)*100:+.1f}%)")
print(f"median moved {flagged['price'].median() - clean_median:+.2f}")

In [ ]:
gap = (df[df.city == 'City 2'].set_index(['date', 'item'])['price']
       - df[df.city == 'City 1'].set_index(['date', 'item'])['price'])
print('City 2 minus City 1, unique gaps:', sorted(gap.unique()))

weekly = df.groupby(['item', 'date'])['price'].mean().unstack('item')
falls = (weekly.diff() < 0).sum()
print('weeks with a week-on-week fall, by item:'); print(falls.to_string())

rng = df.groupby('item')['price'].agg(lambda s: s.max() - s.min())
cv = df.groupby('item')['price'].agg(lambda s: s.std() / s.mean() * 100)
print('range:'); print(rng.round(2).to_string())
print('coefficient of variation %:'); print(cv.round(2).to_string())